# Test Full CV Pipeline on Kaggle

Notebook nay chay lien mach Module 1 segmentation, Module 2 attribute recognition, Module 3 OCR va Module 4 fusion tren mot anh.

Truoc khi chay, attach bon Kaggle Dataset: checkpoint segmentation, artifact Attribute last-block, anh test, va SQLite database da seed. Notebook clone source code tu GitHub vao `/kaggle/working`. PaddleOCR chay trong process GPU rieng de khong xung dot voi PyTorch.

In [ ]:
from pathlib import Path

GIT_REPO_URL = 'https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git'
# Branch nay phai da duoc push va chua ca CV, RAG/DDI va end-to-end orchestration.
GIT_BRANCH = 'FE_Final'
REPO_DIR = Path('/kaggle/working/Multiple-Pill-Recognition-And-Interaction-Safety')

# Sua cac path nay theo ten Kaggle Dataset cua ban.
SEGMENTATION_WEIGHTS = Path(
    '/kaggle/input/datasets/nnphuchcmus/pill-segmentation-model/yolov11m_seg_mediseg_full_finetune_v1.pt'
)
IMAGE_PATH = Path('/kaggle/input/datasets/nnphuchcmus/segment-error/Screenshot 2026-08-27 014510.png')

# Thu muc nay phai chua: best.pt, label_mapping.json, optimal_thresholds.json, model_config.yaml.
ATTRIBUTE_ARTIFACT_DIR = Path(
    '/kaggle/input/datasets/nnphuchcmus/attrubute-artifact/kaggle_uploads/attribute_resnet18_last_blocks_finetune'
)

# Tat ca artifact inference se ghi vao working directory co quyen ghi.
OUTPUT_DIR = Path('/kaggle/working/pill_cv_outputs')

REQUEST_ID = 'req_kaggle_001'
SESSION_ID = 'kaggle_full_cv_test'
IMAGE_ID = IMAGE_PATH.stem

print('REPO_DIR:', REPO_DIR)
print('SEGMENTATION_WEIGHTS:', SEGMENTATION_WEIGHTS)
print('ATTRIBUTE_ARTIFACT_DIR:', ATTRIBUTE_ARTIFACT_DIR)
print('IMAGE_PATH:', IMAGE_PATH)

In [ ]:
# Kaggle Notebook Settings phai bat Internet de clone GitHub.
import subprocess

if not REPO_DIR.is_dir():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', GIT_BRANCH, GIT_REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    # Repo co san co the dang o nhanh cu; cap nhat dung nhanh truoc khi import source.
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', GIT_BRANCH],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'checkout', '--force', '-B', GIT_BRANCH, f'origin/{GIT_BRANCH}'],
        check=True,
    )
    print(f'Repository da duoc chuyen sang nhanh: {GIT_BRANCH}')

active_branch = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'branch', '--show-current'],
    text=True,
).strip()
active_commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'],
    text=True,
).strip()
print('Active repository branch:', active_branch)
print('Active repository commit:', active_commit)
if active_branch != GIT_BRANCH:
    raise RuntimeError(f'Sai branch: {active_branch}; expected {GIT_BRANCH}.')

print('Repository ready:', REPO_DIR)


In [ ]:
# Kernel chinh chay segmentation bang PyTorch/Ultralytics GPU.
# OCR duoc cai theo dung bo package cua PaddleOCR_baseline sau khi segmentation xong.
# --no-deps giu nguyen torch/torchvision CUDA ma Kaggle da cung cap.
%pip install -q --no-deps ultralytics==8.3.253 pyyaml==6.0.2

import sys
import torch

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Torch CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch khong thay GPU. Hay bat GPU accelerator tren Kaggle.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Kiem tra tai nguyen truoc khi import code project.
if not REPO_DIR.is_dir():
    raise FileNotFoundError(
        f'Repo khong ton tai: {REPO_DIR}. Chay lai cell clone va kiem tra Internet setting.'
    )
if not SEGMENTATION_WEIGHTS.is_file():
    raise FileNotFoundError(f'Khong tim thay checkpoint: {SEGMENTATION_WEIGHTS}')
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f'Khong tim thay anh test: {IMAGE_PATH}')
for artifact_name in ('best.pt', 'label_mapping.json', 'optimal_thresholds.json', 'model_config.yaml'):
    artifact_path = ATTRIBUTE_ARTIFACT_DIR / artifact_name
    if not artifact_path.is_file():
        raise FileNotFoundError(f'Khong tim thay Attribute artifact: {artifact_path}')

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pill_safety.cv.segmentation import SegmentationConfig, SegmentationPredictor
from pill_safety.schemas import SegmentationInferenceRequest

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project source:', SRC_DIR)
print('Output directory:', OUTPUT_DIR)


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

with Image.open(IMAGE_PATH) as source:
    image = source.convert('RGB')

plt.figure(figsize=(10, 7))
plt.imshow(image)
plt.title(f'Input image: {IMAGE_PATH.name}')
plt.axis('off')
plt.show()


In [ ]:
# Runtime overrides for Module 1. Set a value to tune it, or None to keep segmentation.yaml.
# These settings affect only this Kaggle session and do not modify source files.
SEGMENTATION_OVERRIDES = {
    # YOLO inference thresholds
    'confidence_threshold': None,
    'iou_threshold': None,
    'mask_threshold': None,
    # Quality gates
    'min_mask_area_ratio': None,
    'max_mask_area_ratio': None,
    'min_component_area_ratio': None,
    'merged_solidity_threshold': None,
    'non_pill_confidence_threshold': None,
    # Crop controls
'bbox_padding_ratio': 0.12, 'color_mask_erosion_ratio': 0.04
}

# Dat ten rieng cho moi lan thu de artifact va ket qua khong de len nhau.
# Khi doi hyperparameter, doi ca ten nay roi Run all de so sanh cong bang.
TUNING_RUN_NAME = 'baseline_margin08_erode004'

SEGMENTATION_OVERRIDES = {
    name: value for name, value in SEGMENTATION_OVERRIDES.items()
    if value is not None
}

# Khong sua YAML: chi doi output runtime cua trial hien tai.
BASE_OUTPUT_DIR = OUTPUT_DIR
OUTPUT_DIR = BASE_OUTPUT_DIR / 'segmentation_tuning' / TUNING_RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Active runtime overrides:', SEGMENTATION_OVERRIDES or 'none; use segmentation.yaml values')
print('Tuning output directory:', OUTPUT_DIR)


In [ ]:
# Tao request truc tiep tu IMAGE_PATH; khong can tao request.json.
from dataclasses import replace
request = SegmentationInferenceRequest(
    request_id=REQUEST_ID,
    session_id=SESSION_ID,
    image_id=IMAGE_ID,
    image_path=str(IMAGE_PATH),
)

config = SegmentationConfig.from_yaml(
    REPO_DIR / 'configs' / 'inference' / 'segmentation.yaml'
).with_weights_path(SEGMENTATION_WEIGHTS).with_output_dir(OUTPUT_DIR)
config = replace(config, **SEGMENTATION_OVERRIDES)
print('Active segmentation config:', {
    name: getattr(config, name) for name in (
        'confidence_threshold', 'iou_threshold', 'mask_threshold',
        'min_mask_area_ratio', 'max_mask_area_ratio',
        'bbox_padding_ratio', 'crop_mask_dilation_ratio',
        'color_mask_erosion_ratio',
    )
})

# Config YAML da dung device=auto: Ultralytics se dung GPU neu Kaggle cap GPU.
predictor = SegmentationPredictor(config=config)
artifacts = predictor.predict_with_artifacts(request)
result = artifacts.output.model_dump(mode='json')

print('Detected instances:', len(result['instances']))
print('Schema JSON:', artifacts.schema_json_path)
print('Overlay:', artifacts.overlay_path)


In [ ]:
# Hien thi overlay va thong tin tong quat.
if artifacts.overlay_path is not None and artifacts.overlay_path.exists():
    with Image.open(artifacts.overlay_path) as source:
        overlay = source.convert('RGB')
    plt.figure(figsize=(12, 8))
    plt.imshow(overlay)
    plt.title('Segmentation overlay')
    plt.axis('off')
    plt.show()

display({
    'image_quality': result['image_quality'],
    'instance_count': len(result['instances']),
})


In [ ]:
# Hien thi clean mask, color crop va shape crop cua tung vien.
instances = result['instances']
if not instances:
    print('Khong phat hien vien nao. Kiem tra checkpoint, confidence_threshold va anh input.')
else:
    figure, axes = plt.subplots(len(instances), 3, figsize=(15, 5 * len(instances)))
    if len(instances) == 1:
        axes = [axes]

    for row, instance in zip(axes, instances):
        with Image.open(instance['mask_path']) as source:
            mask = source.convert('L')
        with Image.open(instance['color_crop_path']) as source:
            color_crop = source.convert('RGB')
        with Image.open(instance['shape_crop_path']) as source:
            shape_crop = source.convert('RGB')

        row[0].imshow(mask, cmap='gray')
        row[0].set_title(f"{instance['instance_id']} mask")
        row[0].axis('off')
        row[1].imshow(color_crop)
        row[1].set_title(
            f"{instance['instance_id']} color crop | conf={instance['segmentation']['confidence']:.3f}"
        )
        row[1].axis('off')
        row[2].imshow(shape_crop)
        row[2].set_title(f"{instance['instance_id']} shape crop")
        row[2].axis('off')

    plt.tight_layout()
    plt.show()

    for instance in instances:
        print('\n', instance['instance_id'])
        display({
            'bbox_xyxy': instance['bbox_xyxy'],
            'segmentation': instance['segmentation'],
            'quality_flags': instance['quality_flags'],
            'mask_path': instance['mask_path'],
            'color_crop_path': instance['color_crop_path'],
            'shape_crop_path': instance['shape_crop_path'],
            'ocr_crop_path': instance['ocr_crop_path'],
            'crop_path': instance['crop_path'],
        })


In [ ]:
# Module 1 JSON nay la input de tao request cho Attribute va OCR o buoc sau.
import json

print(json.dumps(result, indent=2, ensure_ascii=False))
print('\nSaved JSON:', artifacts.schema_json_path)


## Test Module 2 Attribute Recognition from Segmentation Crops

Module 2 nhan `color_crop_path` cho color head va `shape_crop_path` cho shape head. ResNet18 duoc chay tren RGB crop theo dung transform validation da train; production khong dung mask truc tiep. Cell debug tao them masked control crop, chi de do tac dong cua pixel ngoai mask va khong thay doi input production.

In [ ]:
# Khoi tao Module 2 mot lan va tai su dung cho moi pill crop.
from dataclasses import replace

from pill_safety.cv.attribute.config import AttributeInferenceConfig
from pill_safety.cv.attribute.predictors import AttributePredictor
from pill_safety.schemas import AttributeInferenceRequest

attribute_config = AttributeInferenceConfig.from_yaml(
    REPO_DIR / 'configs' / 'inference' / 'attribute.yaml'
).resolve_paths(REPO_DIR)
attribute_config = replace(
    attribute_config,
    weights_path=ATTRIBUTE_ARTIFACT_DIR / 'best.pt',
    label_mapping_path=ATTRIBUTE_ARTIFACT_DIR / 'label_mapping.json',
    color_thresholds_path=ATTRIBUTE_ARTIFACT_DIR / 'optimal_thresholds.json',
    model_config_path=ATTRIBUTE_ARTIFACT_DIR / 'model_config.yaml',
    output_dir=OUTPUT_DIR,
)
attribute_predictor = AttributePredictor(config=attribute_config)

# In raw softmax cua shape de kiem tra model dang chon class index nao.
# Cac bien shape_probabilities/shape_names chi ton tai trong ham debug nay.
import torch

def debug_shape_prediction(predictor, crop_path):
    with Image.open(crop_path) as source:
        image = source.convert('RGB')
    tensor = predictor.transform(image).unsqueeze(0).to(predictor.device)
    with torch.inference_mode():
        logits = predictor.model(tensor, task_type='shape')
        probabilities = torch.softmax(logits, dim=1)[0].detach().cpu()
    shape_names = predictor.label_mapping['shape']
    ranked_probabilities, ranked_indices = torch.sort(
        probabilities, descending=True
    )
    predicted_index = int(ranked_indices[0].item())
    print('RAW SHAPE DEBUG')
    print('probabilities:', {
        shape_names[index]: round(float(probability), 4)
        for index, probability in enumerate(probabilities)
    })
    print('predicted_index:', predicted_index)
    print('predicted_label:', shape_names[predicted_index])
    print('top3:', [
        (shape_names[int(index.item())], round(float(probability.item()), 4))
        for probability, index in zip(
            ranked_probabilities[:3], ranked_indices[:3]
        )
    ])

attribute_outputs = []
attribute_artifacts_by_instance = {}

for instance in artifacts.output.instances:
    attribute_request = AttributeInferenceRequest(
        request_id=artifacts.output.request_id,
        session_id=artifacts.output.session_id,
        image_id=artifacts.output.image_id,
        instance_id=instance.instance_id,
        instance_token=instance.instance_token,
        crop_path=instance.color_crop_path,
        color_crop_path=instance.color_crop_path,
        shape_crop_path=instance.shape_crop_path,
        mask_path=instance.mask_path,
    )
    attribute_artifacts = attribute_predictor.predict_with_artifacts(attribute_request)
    debug_shape_prediction(attribute_predictor, instance.shape_crop_path)
    attribute_outputs.append(attribute_artifacts.output)
    attribute_artifacts_by_instance[instance.instance_id] = attribute_artifacts

print('Attribute completed:', len(attribute_outputs), 'pill(s)')
for attribute_output in attribute_outputs:
    display({
        'instance_id': attribute_output.instance_id,
        'shape': attribute_output.shape.model_dump(mode='json'),
        'color': attribute_output.color.model_dump(mode='json'),
        'scoreline_placeholder': attribute_output.scoreline.model_dump(mode='json'),
        'schema_json_path': str(
            attribute_artifacts_by_instance[attribute_output.instance_id].schema_json_path
        ),
    })


In [ ]:
# DEBUG ONLY: trace doc lap cho shape head; khong ghi de crop, config hay prediction.
# Chay cell Module 1 va Module 2 truoc, sau do doi ID nay de trace tung pill.
TRACE_SHAPE_INSTANCE_ID = 'pill_001'
TRACE_BACKGROUND_VALUE = 127
TRACE_BACKGROUND_TOLERANCE = 3

import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image


def _short_sha256(path: Path) -> str:
    # Xac dinh dung artifact dang duoc nap, tranh trace nham checkpoint hoac label mapping.
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()[:16]


def _rgb_stats(image: Image.Image) -> dict:
    # Tom tat RGB truoc khi qua transform ImageNet.
    array = np.asarray(image.convert('RGB'), dtype=np.uint8)
    return {
        'size_wh': list(image.size),
        'dtype': str(array.dtype),
        'min_rgb': array.min(axis=(0, 1)).astype(int).tolist(),
        'max_rgb': array.max(axis=(0, 1)).astype(int).tolist(),
        'mean_rgb': np.round(array.mean(axis=(0, 1)), 4).tolist(),
        'std_rgb': np.round(array.std(axis=(0, 1)), 4).tolist(),
        'corner_rgb': {
            'top_left': array[0, 0].astype(int).tolist(),
            'top_right': array[0, -1].astype(int).tolist(),
            'bottom_left': array[-1, 0].astype(int).tolist(),
            'bottom_right': array[-1, -1].astype(int).tolist(),
        },
    }


def _canvas_geometry(image: Image.Image, background_value: int, tolerance: int) -> tuple[dict, np.ndarray]:
    # Day la heuristic theo nen canvas 127, khong phai clean mask that cua shape crop.
    array = np.asarray(image.convert('RGB'), dtype=np.int16)
    near_background = np.max(np.abs(array - background_value), axis=2) <= tolerance
    foreground_like = ~near_background
    ys, xs = np.where(foreground_like)
    touches_edge = bool(
        foreground_like[0].any() or foreground_like[-1].any()
        or foreground_like[:, 0].any() or foreground_like[:, -1].any()
    )
    bbox = None if len(xs) == 0 else [
        int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1
    ]
    return {
        'background_value': background_value,
        'background_tolerance': tolerance,
        'near_background_ratio': round(float(near_background.mean()), 6),
        'foreground_like_ratio': round(float(foreground_like.mean()), 6),
        'foreground_like_bbox_xyxy': bbox,
        'foreground_like_touches_canvas_edge': touches_edge,
    }, foreground_like


def _mask_stats(path: Path) -> tuple[dict, np.ndarray]:
    # Mask luu tren dia dong bo voi color/OCR crop; geometry co the khac shape crop.
    with Image.open(path) as source:
        mask = np.asarray(source.convert('L'), dtype=np.uint8) > 0
    ys, xs = np.where(mask)
    bbox = None if len(xs) == 0 else [
        int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1
    ]
    return {
        'size_hw': [int(mask.shape[0]), int(mask.shape[1])],
        'foreground_pixels': int(mask.sum()),
        'foreground_ratio': round(float(mask.mean()), 6),
        'bbox_xyxy': bbox,
    }, mask


instances_by_id = {item.instance_id: item for item in artifacts.output.instances}
if TRACE_SHAPE_INSTANCE_ID not in instances_by_id:
    raise KeyError(
        f'Khong tim thay {TRACE_SHAPE_INSTANCE_ID}. IDs hien co: {sorted(instances_by_id)}'
    )

trace_instance = instances_by_id[TRACE_SHAPE_INSTANCE_ID]
shape_crop_path = Path(trace_instance.shape_crop_path)
shape_masked_control_crop_path = shape_crop_path.with_name(
    shape_crop_path.name.replace('_shape_crop.png', '_shape_masked_control_crop.png')
)
color_crop_path = Path(trace_instance.color_crop_path)
ocr_crop_path = Path(trace_instance.ocr_crop_path)
mask_path = Path(trace_instance.mask_path)
trace_paths = {
    'shape_crop': shape_crop_path,
    'shape_masked_control_crop': shape_masked_control_crop_path,
    'color_crop': color_crop_path,
    'ocr_crop': ocr_crop_path,
    'clean_mask': mask_path,
    'checkpoint': attribute_predictor.config.weights_path,
    'label_mapping': attribute_predictor.config.label_mapping_path,
    'model_config': attribute_predictor.config.model_config_path,
}
missing_paths = [f'{name}: {path}' for name, path in trace_paths.items() if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('Trace thieu file:\n- ' + '\n- '.join(missing_paths))

with Image.open(shape_crop_path) as source:
    trace_shape_image = source.convert('RGB')
with Image.open(shape_masked_control_crop_path) as source:
    trace_shape_masked_control_image = source.convert('RGB')
with Image.open(color_crop_path) as source:
    trace_color_image = source.convert('RGB')
with Image.open(ocr_crop_path) as source:
    trace_ocr_image = source.convert('RGB')

shape_stats = _rgb_stats(trace_shape_image)
shape_masked_control_stats = _rgb_stats(trace_shape_masked_control_image)
color_stats = _rgb_stats(trace_color_image)
ocr_stats = _rgb_stats(trace_ocr_image)
canvas_stats, foreground_like = _canvas_geometry(
    trace_shape_image, TRACE_BACKGROUND_VALUE, TRACE_BACKGROUND_TOLERANCE
)
mask_stats, trace_mask = _mask_stats(mask_path)

# Lap lai chinh xac transform inference: Resize -> ToTensor -> ImageNet Normalize.
transformed_cpu = attribute_predictor.transform(trace_shape_image).detach().cpu()
resized_image = trace_shape_image.resize(
    (attribute_predictor.config.image_size, attribute_predictor.config.image_size),
    Image.Resampling.BILINEAR,
)
resized_array = np.asarray(resized_image, dtype=np.float32) / 255.0
manual_tensor = torch.from_numpy(resized_array).permute(2, 0, 1)
mean = torch.tensor(attribute_predictor.config.normalization_mean).view(3, 1, 1)
std = torch.tensor(attribute_predictor.config.normalization_std).view(3, 1, 1)
manual_transformed = (manual_tensor - mean) / std
transform_max_abs_difference = float(
    (transformed_cpu - manual_transformed).abs().max().item()
)

shape_tensor = transformed_cpu.unsqueeze(0).to(attribute_predictor.device)
with torch.inference_mode():
    first_output = attribute_predictor.model(shape_tensor)
    first_logits = first_output[0] if isinstance(first_output, tuple) else first_output
    second_output = attribute_predictor.model(shape_tensor)
    second_logits = second_output[0] if isinstance(second_output, tuple) else second_output
    shape_probabilities = torch.softmax(first_logits, dim=1)[0].detach().cpu()
    forward_repeat_max_abs_difference = float(
        (first_logits - second_logits).abs().max().item()
    )

shape_names = list(attribute_predictor.label_mapping['shape'])
if tuple(first_logits.shape) != (1, len(shape_names)):
    raise RuntimeError(
        f'Shape logits sai kich thuoc: {tuple(first_logits.shape)}; expected (1, {len(shape_names)})'
    )

ranked_probabilities, ranked_indices = torch.sort(shape_probabilities, descending=True)
top_rows = [
    {
        'rank': rank + 1,
        'class_index': int(index.item()),
        'label': shape_names[int(index.item())],
        'logit': round(float(first_logits[0, int(index.item())].item()), 8),
        'softmax_probability': round(float(probability.item()), 8),
    }
    for rank, (probability, index) in enumerate(
        zip(ranked_probabilities, ranked_indices)
    )
]
top1_top2_margin = (
    float(ranked_probabilities[0].item() - ranked_probabilities[1].item())
    if len(ranked_probabilities) > 1 else None
)

# Reproduce the exact 0/90/180/270-degree TTA used by AttributePredictor.
# This separates a genuinely unstable orientation response from a stale notebook/runtime.
tta_angles = (0, 90, 180, 270)
tta_images = [
    trace_shape_image if angle == 0 else trace_shape_image.rotate(
        angle, resample=Image.Resampling.BILINEAR, expand=True
    )
    for angle in tta_angles
]
tta_tensor = torch.stack([
    attribute_predictor.transform(image).to(attribute_predictor.device)
    for image in tta_images
])
with torch.inference_mode():
    tta_output = attribute_predictor.model(tta_tensor)
    per_rotation_logits = (
        tta_output[0]
        if isinstance(tta_output, tuple)
        else attribute_predictor.model(tta_tensor, task_type='shape')
    )
    runtime_tta_logits = attribute_predictor._shape_logits_tta(trace_shape_image)
    tta_mean_logits = per_rotation_logits.mean(dim=0, keepdim=True)
    tta_probabilities = torch.softmax(tta_mean_logits, dim=1)[0].detach().cpu()
    per_rotation_probabilities = torch.softmax(per_rotation_logits, dim=1).detach().cpu()
    tta_runtime_max_abs_difference = float(
        (runtime_tta_logits - tta_mean_logits).abs().max().item()
    )
    tta_angle_zero_max_abs_difference = float(
        (per_rotation_logits[0:1] - first_logits).abs().max().item()
    )

tta_ranked_probabilities, tta_ranked_indices = torch.sort(tta_probabilities, descending=True)
tta_top_rows = [
    {
        'rank': rank + 1,
        'class_index': int(index.item()),
        'label': shape_names[int(index.item())],
        'mean_logit': round(float(tta_mean_logits[0, int(index.item())].item()), 8),
        'softmax_probability': round(float(probability.item()), 8),
    }
    for rank, (probability, index) in enumerate(
        zip(tta_ranked_probabilities, tta_ranked_indices)
    )
]
tta_top1_top2_margin = (
    float(tta_ranked_probabilities[0].item() - tta_ranked_probabilities[1].item())
    if len(tta_ranked_probabilities) > 1 else None
)
per_rotation_rows = []
for row_index, angle in enumerate(tta_angles):
    probabilities = per_rotation_probabilities[row_index]
    top_probability, top_index = torch.max(probabilities, dim=0)
    per_rotation_rows.append({
        'angle_degrees': int(angle),
        'top1_index': int(top_index.item()),
        'top1_label': shape_names[int(top_index.item())],
        'top1_probability': round(float(top_probability.item()), 8),
        'probabilities': {
            shape_names[index]: round(float(probability.item()), 8)
            for index, probability in enumerate(probabilities)
        },
    })

# A/B control: identical ROI and orientation, but remove every pixel outside
# the clean shape mask. This is diagnostic only; production still uses shape_crop.
with torch.inference_mode():
    masked_control_tta_logits = attribute_predictor._shape_logits_tta(
        trace_shape_masked_control_image
    )
    masked_control_probabilities = torch.softmax(
        masked_control_tta_logits, dim=1
    )[0].detach().cpu()
masked_control_ranked_probabilities, masked_control_ranked_indices = torch.sort(
    masked_control_probabilities, descending=True
)
masked_control_top_rows = [
    {
        'rank': rank + 1,
        'class_index': int(index.item()),
        'label': shape_names[int(index.item())],
        'mean_logit': round(float(masked_control_tta_logits[0, int(index.item())].item()), 8),
        'softmax_probability': round(float(probability.item()), 8),
    }
    for rank, (probability, index) in enumerate(
        zip(masked_control_ranked_probabilities, masked_control_ranked_indices)
    )
]
masked_control_top1_top2_margin = (
    float(masked_control_ranked_probabilities[0].item() - masked_control_ranked_probabilities[1].item())
    if len(masked_control_ranked_probabilities) > 1 else None
)
masked_control_top_label = masked_control_top_rows[0]['label']
shape_label_changes_when_masked = masked_control_top_label != tta_top_rows[0]['label']
pipeline_attribute = next(
    (output for output in attribute_outputs if output.instance_id == TRACE_SHAPE_INSTANCE_ID),
    None,
)
pipeline_shape = (
    None
    if pipeline_attribute is None
    else pipeline_attribute.shape.model_dump(mode='json')
)
tta_top_label = tta_top_rows[0]['label']
pipeline_label = None if pipeline_shape is None else pipeline_shape.get('label')
pipeline_matches_runtime_tta = (
    pipeline_label is None
    or str(pipeline_label).upper() == str(tta_top_label).upper()
)
rotation_top_labels = sorted({row['top1_label'] for row in per_rotation_rows})
shape_head_parameters = [
    {
        'name': name,
        'shape': list(parameter.shape),
        'l2_norm': round(float(parameter.detach().float().norm().cpu().item()), 8),
    }
    for name, parameter in attribute_predictor.model.named_parameters()
    if 'shape_head' in name or 'fc_shape' in name
]

warnings = []
if attribute_predictor.model.training:
    warnings.append('Model dang o train mode; inference phai dung eval mode.')
if transform_max_abs_difference > 1e-6:
    warnings.append('Transform runtime khac Resize/ToTensor/ImageNet Normalize.')
if forward_repeat_max_abs_difference > 1e-6:
    warnings.append('Cung tensor nhung logits khac nhau giua hai lan forward.')
if top1_top2_margin is not None and top1_top2_margin < 0.05:
    warnings.append('Top-1 va top-2 qua sat; shape prediction khong on dinh.')
if tta_runtime_max_abs_difference > 1e-6:
    warnings.append('Trace TTA thu cong khac AttributePredictor._shape_logits_tta().')
if tta_angle_zero_max_abs_difference > 1e-4:
    warnings.append('Logits 0 do trong TTA khac dang ke single-forward; can kiem tra runtime/model.')
if len(rotation_top_labels) > 1:
    warnings.append(
        'Cung ROI nhung top-1 doi theo goc xoay: ' + ', '.join(rotation_top_labels)
    )
if tta_top1_top2_margin is not None and tta_top1_top2_margin < 0.05:
    warnings.append('TTA top-1 va top-2 qua sat; ket qua shape sau TTA khong on dinh.')
if shape_label_changes_when_masked:
    warnings.append(
        'Nhan shape doi khi bo pixel ngoai mask: co dau hieu background/ROI leakage.'
    )
if not pipeline_matches_runtime_tta:
    warnings.append('Pipeline output khac TTA trace; kernel/co de co the dang chay code cu.')
if canvas_stats['foreground_like_touches_canvas_edge']:
    warnings.append('Heuristic nen cho thay ROI cham bien canvas; can kiem tra crop/margin.')
if trace_mask.shape != np.asarray(trace_shape_image).shape[:2]:
    warnings.append('Saved mask khong dong bo pixel voi shape crop; expected khi shape crop co flow rieng.')

trace_report = {
    'instance_id': TRACE_SHAPE_INSTANCE_ID,
    'segmentation': {
        'bbox_xyxy': list(trace_instance.bbox_xyxy),
        'segmentation_confidence': float(trace_instance.segmentation.confidence),
        'quality_flags': list(trace_instance.quality_flags),
    },
    'paths': {name: str(path) for name, path in trace_paths.items()},
    'artifacts': {
        name: {'bytes': int(path.stat().st_size), 'sha256_16': _short_sha256(path)}
        for name, path in trace_paths.items()
        if name in {'checkpoint', 'label_mapping', 'model_config'}
    },
    'model_runtime': {
        'model_class': type(attribute_predictor.model).__name__,
        'device': str(attribute_predictor.device),
        'training_mode': bool(attribute_predictor.model.training),
        'image_size': int(attribute_predictor.config.image_size),
        'normalization_mean': list(attribute_predictor.config.normalization_mean),
        'normalization_std': list(attribute_predictor.config.normalization_std),
        'shape_head_parameters': shape_head_parameters,
    },
    'label_mapping_index_to_name': {
        str(index): name for index, name in enumerate(shape_names)
    },
    'input_rgb': {
        'shape_crop': shape_stats,
        'shape_masked_control_crop': shape_masked_control_stats,
        'color_crop': color_stats,
        'ocr_crop': ocr_stats,
        'shape_canvas_heuristic': canvas_stats,
        'saved_clean_mask': mask_stats,
    },
    'transform': {
        'tensor_shape': list(transformed_cpu.shape),
        'tensor_min': round(float(transformed_cpu.min().item()), 8),
        'tensor_max': round(float(transformed_cpu.max().item()), 8),
        'tensor_channel_mean': [
            round(float(value), 8) for value in transformed_cpu.mean(dim=(1, 2))
        ],
        'manual_vs_runtime_max_abs_difference': transform_max_abs_difference,
    },
    'shape_head': {
        'logits_shape': list(first_logits.shape),
        'forward_repeat_max_abs_difference': forward_repeat_max_abs_difference,
        'single_orientation_0deg': {
            'ranked_classes': top_rows,
            'top1_top2_probability_margin': top1_top2_margin,
        },
        'runtime_tta': {
            'angles_degrees': list(tta_angles),
            'manual_vs_predictor_max_abs_difference': tta_runtime_max_abs_difference,
            'angle_zero_vs_single_forward_max_abs_difference': tta_angle_zero_max_abs_difference,
            'per_rotation': per_rotation_rows,
            'rotation_top_labels': rotation_top_labels,
            'mean_logits_ranked_classes': tta_top_rows,
            'top1_top2_probability_margin': tta_top1_top2_margin,
        },
        'masked_shape_control_tta': {
            'purpose': 'A/B diagnostic only; not the production input',
            'label_changes_from_production_crop': shape_label_changes_when_masked,
            'mean_logits_ranked_classes': masked_control_top_rows,
            'top1_top2_probability_margin': masked_control_top1_top2_margin,
        },
    },
    'pipeline_output': pipeline_shape,
    'pipeline_matches_runtime_tta': pipeline_matches_runtime_tta,
    'warnings': warnings,
}

trace_directory = OUTPUT_DIR / 'debug' / 'shape_trace'
trace_directory.mkdir(parents=True, exist_ok=True)
trace_json_path = trace_directory / f'{TRACE_SHAPE_INSTANCE_ID}_trace.json'
trace_json_path.write_text(
    json.dumps(trace_report, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('=== SHAPE TRACE COMPLETE ===')
print('instance_id:', TRACE_SHAPE_INSTANCE_ID)
print('shape crop:', shape_crop_path)
print('checkpoint sha256_16:', trace_report['artifacts']['checkpoint']['sha256_16'])
print('label mapping:', trace_report['label_mapping_index_to_name'])
print('single 0-degree top1/top2 margin:', None if top1_top2_margin is None else round(top1_top2_margin, 8))
print('single 0-degree ranked logits + softmax:')
for row in top_rows:
    print(row)
print('TTA top1/top2 margin:', None if tta_top1_top2_margin is None else round(tta_top1_top2_margin, 8))
print('TTA mean-logit ranked classes:')
for row in tta_top_rows:
    print(row)
print('per-rotation TTA top1:')
for row in per_rotation_rows:
    print({key: row[key] for key in ('angle_degrees', 'top1_label', 'top1_probability')})
print('manual TTA vs predictor max abs diff:', tta_runtime_max_abs_difference)
print('pipeline matches runtime TTA:', pipeline_matches_runtime_tta)
print('A/B masked-shape control top1/top2 margin:', None if masked_control_top1_top2_margin is None else round(masked_control_top1_top2_margin, 8))
print('A/B masked-shape control ranked classes:')
for row in masked_control_top_rows:
    print(row)
print('label changes when outside-mask pixels are removed:', shape_label_changes_when_masked)
print('pipeline shape:', trace_report['pipeline_output'])
print('warnings:', warnings or ['none'])
print('trace JSON:', trace_json_path)

figure, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0, 0].imshow(trace_shape_image)
axes[0, 0].set_title('shape_crop passed to shape head')
axes[0, 1].imshow(resized_image)
axes[0, 1].set_title(f'Resize to {attribute_predictor.config.image_size}x{attribute_predictor.config.image_size}')
axes[0, 2].imshow(foreground_like, cmap='magma')
axes[0, 2].set_title('heuristic: pixels not near canvas background')
axes[1, 0].imshow(trace_color_image)
axes[1, 0].set_title('color_crop for comparison only')
axes[1, 1].imshow(trace_ocr_image)
axes[1, 1].set_title('ocr_crop for comparison only')
axes[1, 2].imshow(trace_mask, cmap='gray')
axes[1, 2].set_title('saved clean mask: color/OCR coordinate system')
for axis in axes.flat:
    axis.axis('off')
plt.tight_layout()
plt.show()

figure, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(trace_shape_image)
axes[0].set_title(f"Production crop → {tta_top_rows[0]['label']} ({tta_top_rows[0]['softmax_probability']:.1%})")
axes[1].imshow(trace_shape_masked_control_image)
axes[1].set_title(f"Masked A/B control → {masked_control_top_rows[0]['label']} ({masked_control_top_rows[0]['softmax_probability']:.1%})")
for axis in axes:
    axis.axis('off')
plt.suptitle('A/B: effect of removing pixels outside the clean shape mask')
plt.tight_layout()
plt.show()

figure, axes = plt.subplots(1, len(tta_images), figsize=(16, 4))
for axis, angle, image, row in zip(axes, tta_angles, tta_images, per_rotation_rows):
    axis.imshow(image)
    axis.set_title(
        f"{angle}° → {row['top1_label']} ({row['top1_probability']:.1%})"
    )
    axis.axis('off')
plt.suptitle('Shape prediction for each TTA rotation')
plt.tight_layout()
plt.show()


## Test Module 3 OCR from Segmentation Crops

Cac cell ben duoi dung truc tiep `ocr_crop_path` va clean `mask_path` do Module 1 vua sinh ra. OCR khong can upload lai anh va khong can tao file request JSON.

In [ ]:
# Ghi de cac thong so cua OCR giong nhu Segmentation
OCR_OVERRIDES = {
'det_db_thresh': 0.3, 'det_db_unclip_ratio': 2.0, 'min_usable_confidence': 0.3
}

In [ ]:
# Tao PaddleOCR GPU environment rieng de khong thay NCCL/CUDA cua PyTorch.
# Cell nay chi cai mot lan; cac lan chay sau se tai su dung OCR_VENV_DIR.
import gc
import subprocess

# Module 1 va Module 2 da hoan tat; giai phong hai model PyTorch de PaddleOCR co du VRAM.
if 'predictor' in globals():
    del predictor
if 'attribute_predictor' in globals():
    del attribute_predictor
gc.collect()
torch.cuda.empty_cache()
print('Released YOLO model before starting PaddleOCR GPU process.')

OCR_VENV_DIR = Path('/kaggle/working/paddleocr_gpu_venv')
OCR_PYTHON = OCR_VENV_DIR / 'bin' / 'python'
OCR_READY_FILE = OCR_VENV_DIR / '.ocr_environment_ready'

def run_checked(command, label):
    print(f'\n>>> {label}')
    print(' '.join(str(part) for part in command))
    # Khong capture stdout/stderr de Kaggle hien tien do pip theo thoi gian thuc.
    completed = subprocess.run(command)
    if completed.returncode != 0:
        raise RuntimeError(f'{label} failed (exit={completed.returncode})')
    return completed

def install_ocr_environment():
    # Kaggle /usr/bin/python3 khong co ensurepip, nen dung virtualenv thay cho stdlib venv.
    # --clear tu sua thu muc bi tao do dang tu lan chay truoc.
    virtualenv_check = subprocess.run(
        [sys.executable, '-m', 'virtualenv', '--version'],
        text=True, capture_output=True,
    )
    if virtualenv_check.returncode != 0:
        run_checked(
            [
                sys.executable, '-m', 'pip', 'install',
                'virtualenv==20.31.2',
            ],
            'Install virtualenv for the OCR subprocess',
        )
    run_checked(
        [sys.executable, '-m', 'virtualenv', '--clear', str(OCR_VENV_DIR)],
        'Create isolated PaddleOCR environment',
    )
    # Paddle imports setuptools during startup; virtualenv may not seed it on Kaggle.
    run_checked(
        [
            str(OCR_PYTHON), '-m', 'pip', 'install',
            '--upgrade', 'pip', 'setuptools', 'wheel',
        ],
        'Bootstrap OCR environment packaging tools',
    )
    run_checked(
        [
            str(OCR_PYTHON), '-m', 'pip', 'install',
            'paddlepaddle-gpu==3.0.0',
            '--index-url', 'https://www.paddlepaddle.org.cn/packages/stable/cu118/',
        ],
        'Install Paddle GPU',
    )
    run_checked(
        [
            str(OCR_PYTHON), '-m', 'pip', 'install',
            'paddleocr==3.0.3', 'paddlex==3.0.3',
            'langchain==0.3.27', 'langchain-community==0.3.27',
            'langchain-text-splitters==0.3.9',
            'opencv-python-headless==4.10.0.84', 'numpy==1.26.4',
            'pillow==11.0.0', 'pydantic==2.9.2', 'pyyaml==6.0.2',
        ],
        'Install PaddleOCR dependencies',
    )


OCR_CHECK_COMMAND = [
    str(OCR_PYTHON), '-c',
    (
        "from importlib.metadata import version; "
        "import setuptools; import paddle, paddleocr, paddlex; "
        "from langchain.docstore.document import Document; "
        "assert version('paddlepaddle-gpu') == '3.0.0'; "
        "assert version('paddleocr') == '3.0.3'; "
        "assert version('paddlex') == '3.0.3'; "
        "assert version('langchain') == '0.3.27'; "
        "assert paddle.device.is_compiled_with_cuda(); "
        "paddle.set_device('gpu:0'); "
        "print('Paddle', paddle.__version__); "
        "print('PaddleOCR', paddleocr.__version__); "
        "print('PaddleX', version('paddlex')); "
        "print('Paddle device', paddle.get_device())"
    ),
]

def check_ocr_environment():
    if not OCR_PYTHON.is_file():
        return None
    return subprocess.run(
        OCR_CHECK_COMMAND, text=True, capture_output=True
    )


ocr_env_check = check_ocr_environment() if OCR_READY_FILE.is_file() else None
if ocr_env_check is None or ocr_env_check.returncode != 0:
    if OCR_READY_FILE.exists():
        OCR_READY_FILE.unlink()
    print('Building or repairing isolated PaddleOCR GPU environment...')
    install_ocr_environment()
    ocr_env_check = check_ocr_environment()

if ocr_env_check is None:
    raise RuntimeError(f'OCR Python was not created: {OCR_PYTHON}')
print(ocr_env_check.stdout)
if ocr_env_check.returncode != 0:
    raise RuntimeError(
        'PaddleOCR GPU environment check failed after rebuild:\n'
        f'STDOUT:\n{ocr_env_check.stdout}\nSTDERR:\n{ocr_env_check.stderr}'
    )
OCR_READY_FILE.touch()
print('OCR GPU environment: READY')


In [ ]:
# OCR chay trong process Paddle GPU rieng; kernel PyTorch khong import Paddle.
OCR_OUTPUT_DIR = OUTPUT_DIR / 'predictions' / 'ocr'
OCR_REQUEST_DIR = OUTPUT_DIR / 'requests' / 'ocr'
OCR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OCR_REQUEST_DIR.mkdir(parents=True, exist_ok=True)
OCR_RUNNER = REPO_DIR / 'inference' / 'cv_ocr' / 'run_ocr.py'
OCR_CONFIG_PATH = REPO_DIR / 'configs' / 'inference' / 'ocr.yaml'

if not OCR_RUNNER.is_file():
    raise FileNotFoundError(f'Khong tim thay OCR runner: {OCR_RUNNER}')
print('OCR output directory:', OCR_OUTPUT_DIR)


In [ ]:
# Tao request JSON tu crop/mask Module 1 va chay OCR tren Paddle GPU process.
# Quy tac nay phai trung voi _safe_directory_name() trong OCRPredictor.
import re

def ocr_artifact_directory_name(value):
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('._')
    return safe or 'pill'

ocr_results_by_instance = {}
ocr_artifact_paths_by_instance = {}

import yaml

with open(OCR_CONFIG_PATH, 'r') as f:
    ocr_cfg = yaml.safe_load(f)

if 'OCR_OVERRIDES' in globals():
    if 'det_db_thresh' in OCR_OVERRIDES:
        ocr_cfg['model']['det_db_thresh'] = float(OCR_OVERRIDES['det_db_thresh'])
    if 'det_db_unclip_ratio' in OCR_OVERRIDES:
        ocr_cfg['model']['det_db_unclip_ratio'] = float(OCR_OVERRIDES['det_db_unclip_ratio'])
    if 'min_usable_confidence' in OCR_OVERRIDES:
        ocr_cfg['pipeline']['min_usable_confidence'] = float(OCR_OVERRIDES['min_usable_confidence'])

active_ocr_config_path = OCR_REQUEST_DIR / 'active_ocr_override.yaml'
with open(active_ocr_config_path, 'w') as f:
    yaml.dump(ocr_cfg, f)

if not result['instances']:
    print('Khong co segmentation instance, bo qua OCR.')
else:
    for instance in result['instances']:
        instance_id = instance['instance_id']
        ocr_request_payload = {
            'request_id': result['request_id'],
            'session_id': result['session_id'],
            'image_id': result['image_id'],
            'instance_id': instance_id,
            'instance_token': instance['instance_token'],
            'crop_path': instance['ocr_crop_path'],
            'mask_path': instance['mask_path'],
        }
        request_path = OCR_REQUEST_DIR / f'{instance_id}_request.json'
        request_path.write_text(
            json.dumps(ocr_request_payload, indent=2, ensure_ascii=False),
            encoding='utf-8',
        )

        print(f'Running OCR on Paddle GPU: {instance_id}')
        completed = subprocess.run(
            [
                str(OCR_PYTHON), str(OCR_RUNNER),
                '--request', str(request_path),
                '--config', str(active_ocr_config_path),
                '--output-dir', str(OCR_OUTPUT_DIR),
            ],
            cwd=str(REPO_DIR), text=True, capture_output=True,
        )
        if completed.returncode != 0:
            raise RuntimeError(
                f'OCR failed for {instance_id}:\nSTDOUT:\n{completed.stdout}\n'
                f'STDERR:\n{completed.stderr}'
            )
        instance_dir = (
            OCR_OUTPUT_DIR
            / ocr_artifact_directory_name(result['request_id'])
            / ocr_artifact_directory_name(result['image_id'])
            / ocr_artifact_directory_name(instance_id)
        )
        artifact_paths = {
            'schema': instance_dir / f'{ocr_artifact_directory_name(instance_id)}_ocr_schema.json',
            'debug': instance_dir / f'{ocr_artifact_directory_name(instance_id)}_final_result.json',
            'overlay': instance_dir / f'{ocr_artifact_directory_name(instance_id)}_final_overlay.jpg',
        }
        if not artifact_paths['schema'].is_file():
            raise FileNotFoundError(
                f'OCR process thanh cong nhung khong tao schema: {artifact_paths["schema"]}'
            )
        ocr_result = json.loads(artifact_paths['schema'].read_text(encoding='utf-8'))
        ocr_results_by_instance[instance_id] = ocr_result
        ocr_artifact_paths_by_instance[instance_id] = artifact_paths

print('OCR completed:', len(ocr_results_by_instance), 'pill(s)')


In [ ]:
# Hien thi crop, final overlay, final answer va scoreline cua tung vien.
for instance in result['instances']:
    instance_id = instance['instance_id']
    ocr_result = ocr_results_by_instance.get(instance_id)
    artifact_paths = ocr_artifact_paths_by_instance.get(instance_id)
    if ocr_result is None or artifact_paths is None:
        continue

    with Image.open(instance['ocr_crop_path']) as source:
        crop = source.convert('RGB')

    figure, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(crop)
    axes[0].set_title(f'{instance_id} crop from Module 1')
    axes[0].axis('off')

    if artifact_paths['overlay'].is_file():
        with Image.open(artifact_paths['overlay']) as source:
            overlay = source.convert('RGB')
        axes[1].imshow(overlay)
        axes[1].set_title('Module 3 final OCR overlay')
    else:
        axes[1].text(0.5, 0.5, 'No OCR overlay: no usable text', ha='center', va='center')
        axes[1].set_title('Module 3 final OCR overlay')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    display({
        'instance_id': instance_id,
        'imprint_visibility': ocr_result['imprint_visibility'],
        'final_text': ocr_result['imprint']['raw'],
        'final_confidence': ocr_result['imprint']['confidence'],
        'scoreline': ocr_result['scoreline'],
        'normalized_candidates': ocr_result['imprint']['normalized_candidates'],
        'schema_json_path': str(artifact_paths['schema']),
        'debug_json_path': str(artifact_paths['debug']),
    })


In [ ]:
# In Module 3 JSON de kiem tra schema va instance_token truoc khi fusion Module 4.
ocr_outputs = []
for instance_id, ocr_result in ocr_results_by_instance.items():
    ocr_outputs.append(ocr_result)
    print(f'\n===== Module 3 OCR JSON: {instance_id} =====')
    print(json.dumps(ocr_result, indent=2, ensure_ascii=False))

print('\nOCR schema artifacts:')
for artifact_paths in ocr_artifact_paths_by_instance.values():
    print(artifact_paths['schema'])


## Module 4: Fuse CV Outputs

Cell nay chi fusion cac JSON da co. No kiem tra `request_id`, `image_id`, `instance_id` va `instance_token`, sau do thay placeholder scoreline cua Module 2 bang ket qua OCR cua Module 3.

In [ ]:
from pill_safety.cv.pipeline import CVPipelineAssembler, CVPipelineConfig
from pill_safety.schemas import CVPipelineInput

pipeline_config = CVPipelineConfig.from_yaml(
    REPO_DIR / 'configs' / 'inference' / 'cv_pipeline.yaml'
).with_output_dir(OUTPUT_DIR)
pipeline_artifacts = CVPipelineAssembler(config=pipeline_config).predict_with_artifacts(
    CVPipelineInput(
        segmentation_output=artifacts.output,
        attribute_outputs=attribute_outputs,
        ocr_outputs=ocr_outputs,
    )
)
cv_output = pipeline_artifacts.output.model_dump(mode='json')

print('CV fusion completed:', len(cv_output['pills']), 'pill(s)')
print('Final CV schema:', pipeline_artifacts.schema_json_path)
print(json.dumps(cv_output, indent=2, ensure_ascii=False))


In [ ]:
# CELL ĐÁNH GIÁ (EVALUATION) TUNING
import json
import difflib
from pathlib import Path

# Nguon Ground Truth ma user da label
# Chu y: pill_01 thanh pill_001 theo chuan output cua YOLO
ground_truth_labels = {
    'pill_003': '54868-5095',
    'pill_004': '54348-857',
    'pill_001': '61919-448',
    'pill_002': '72789-401'
}

# Duong dan toi file JSON. Neu chay tren Kaggle, hay tao Dataset tu file nay va doi duong dan.
json_path = Path(r'/kaggle/input/datasets/nnphuchcmus/drug-label/drug_appearances.json')

if not json_path.exists():
    print(f'KHONG TIM THAY JSON TAI: {json_path}. Vui long kiem tra lai duong dan.')
else:
    with open(json_path, 'r', encoding='utf-8') as f:
        drug_db = json.load(f)
    
    drug_db_dict = {item['product_code']: item for item in drug_db}
    
    total_correct_shape = 0
    total_correct_color = 0
    total_correct_imprint = 0
    total_pills = len(ground_truth_labels)
    
    print('=' * 40)
    print('KET QUA DANH GIA KHI TUNE HYPERPARAMETERS')
    print('=' * 40)
    
    for pill in cv_output.get('pills', []):
        instance_id = pill['instance_id']
        if instance_id not in ground_truth_labels:
            print(f'Bo qua {instance_id} vi khong nam trong tap ground truth dac dinh.')
            continue
            
        gt_code = ground_truth_labels[instance_id]
        gt_info = drug_db_dict.get(gt_code)
        if not gt_info:
            print(f'Loi: Khong tim thay code {gt_code} trong file JSON.')
            continue
            
        pred_shape = str(pill.get('shape', {}).get('label', '')).upper()
        pred_color = str(pill.get('color', {}).get('primary', '')).upper()
        
        pred_imprint = str(pill.get('imprint', {}).get('raw', '')).upper()
        if pred_imprint == 'UNKNOWN':
             pred_imprint = ''
        
        gt_shape = str(gt_info.get('shape') or '').upper()
        gt_color = str(gt_info.get('primary_color') or '').upper()
        gt_imprint = str(gt_info.get('imprint_normalized') or '').upper()
        
        is_shape_ok = 1.0 if pred_shape == gt_shape else 0.0
        is_color_ok = 1.0 if pred_color == gt_color else 0.0
        
        score_i = 0.0
        if gt_imprint == pred_imprint:
            score_i = 1.0
        elif gt_imprint and pred_imprint:
            sm = difflib.SequenceMatcher(None, gt_imprint, pred_imprint)
            match = sm.find_longest_match(0, len(gt_imprint), 0, len(pred_imprint))
            score_i = match.size / len(gt_imprint)
            
        total_correct_shape += is_shape_ok
        total_correct_color += is_color_ok
        total_correct_imprint += score_i
        
        print(f'\n[{instance_id}] So sanh voi GT: {gt_code}')
        print(f'   Shape   : {pred_shape:<12} | GT: {gt_shape:<12} -> {"PASS" if is_shape_ok == 1.0 else "FAIL"}')
        print(f'   Color   : {pred_color:<12} | GT: {gt_color:<12} -> {"PASS" if is_color_ok == 1.0 else "FAIL"}')
        print(f'   Imprint : {pred_imprint:<12} | GT: {gt_imprint:<12} -> Score: {score_i:.2f}')
        
    print('\n' + '=' * 40)
    print(f'Tong diem Shape   : {total_correct_shape:.2f}/{total_pills}')
    print(f'Tong diem Color   : {total_correct_color:.2f}/{total_pills}')
    print(f'Tong diem Imprint : {total_correct_imprint:.2f}/{total_pills}')
    print('-' * 40)
    final_score = total_correct_shape + total_correct_color + total_correct_imprint
    print(f'\nFINAL TUNING SCORE: {final_score:.2f} / {total_pills * 3}')


## Module 5-8: Retrieval, DDI, and Grounded LLM Report

Attach a Kaggle dataset containing exactly one seeded `pill_safety.db`. The next cell discovers it automatically, then consumes the `cv_output_v1` generated by Module 4, identifies only accepted candidates, checks DDI, and builds a grounded fallback report.

In [ ]:
# Tu dong tim SQLite database da attach vao Kaggle Input.
# Dataset can nam trong nhieu cap thu muc; file bat buoc co ten pill_safety.db.
DATABASE_FILE_NAME = 'pill_safety.db'
database_candidates = sorted(Path('/kaggle/input').rglob(DATABASE_FILE_NAME))
if len(database_candidates) == 0:
    raise FileNotFoundError(
        f'Khong tim thay {DATABASE_FILE_NAME} trong /kaggle/input. Attach Kaggle dataset chua SQLite database da seed.'
    )
if len(database_candidates) > 1:
    found = '\n'.join(str(path) for path in database_candidates)
    raise RuntimeError(
        f'Tim thay nhieu file {DATABASE_FILE_NAME}; chi attach mot database dataset de tranh dung nham:\n{found}'
    )
DATABASE_PATH = database_candidates[0]
if DATABASE_PATH.stat().st_size == 0:
    raise ValueError(f'Database rong: {DATABASE_PATH}')
DATABASE_URL = f'sqlite:///{DATABASE_PATH.as_posix()}'
MARKET = 'US'
KNOWN_DRUG_NAMES = []
LLM_PROVIDER = 'fallback'

print('SQLite database:', DATABASE_PATH)
print('DATABASE_URL:', DATABASE_URL)
print('LLM_PROVIDER:', LLM_PROVIDER)

import os
import pandas as pd
from pill_safety.core.config import get_settings

# Set environment before importing SessionLocal because its engine is created at import time.
os.environ['DATABASE_URL'] = DATABASE_URL
os.environ['LLM_PROVIDER'] = LLM_PROVIDER
get_settings.cache_clear()

from pill_safety.database.session import SessionLocal
from pill_safety.rag.orchestration import EndToEndPostCvPipeline

END_TO_END_DIR = OUTPUT_DIR / 'reports' / cv_output['request_id']
db = SessionLocal()
try:
    end_to_end_artifacts = EndToEndPostCvPipeline.from_database_session(
        db,
        llm_provider=LLM_PROVIDER,
    ).run_with_artifacts(
        cv_output,
        output_dir=END_TO_END_DIR,
        market=MARKET,
        known_drug_names=KNOWN_DRUG_NAMES,
    )
finally:
    db.close()

end_to_end_result = end_to_end_artifacts.output
print('End-to-end completed:', len(end_to_end_result['pill_summary']), 'pill(s)')
for name, path in end_to_end_artifacts.paths.items():
    print(f'{name}: {path}')

display(pd.DataFrame(end_to_end_result['pill_summary']))
print('\n===== Grounded LLM report =====')
print(end_to_end_result['llm_report']['formatted_report_text'])


In [ ]:
# Luu ket qua cua trial hien tai de so sanh cac SEGMENTATION_OVERRIDES.
# Chi chay cell nay SAU cell Module 5-8 da tao end_to_end_result.
from datetime import datetime, timezone

summary_frame = pd.DataFrame(end_to_end_result['pill_summary'])
status_counts = summary_frame['identification_status'].value_counts()
score_series = pd.to_numeric(
    summary_frame.get('top_candidate_score'), errors='coerce'
)

trial_record = {
    'run_name': TUNING_RUN_NAME,
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    'detected_instances': len(result['instances']),
    'identified_count': int(status_counts.get('identified', 0)),
    'ambiguous_count': int(status_counts.get('ambiguous', 0)),
    'unknown_count': int(status_counts.get('unknown', 0)),
    'insufficient_visual_evidence_count': int(
        status_counts.get('insufficient_visual_evidence', 0)
    ),
    'mean_top_candidate_score': round(float(score_series.mean()), 4)
    if score_series.notna().any() else None,
    'overrides_json': json.dumps(
        SEGMENTATION_OVERRIDES, sort_keys=True, ensure_ascii=False
    ),
    'output_dir': str(OUTPUT_DIR),
}

TUNING_HISTORY_PATH = BASE_OUTPUT_DIR / 'segmentation_tuning_history.csv'
if TUNING_HISTORY_PATH.is_file():
    tuning_history = pd.read_csv(TUNING_HISTORY_PATH)
    # Re-run cung ten thi thay the dong cu, tranh dem trung mot trial.
    tuning_history = tuning_history[
        tuning_history['run_name'] != TUNING_RUN_NAME
    ]
else:
    tuning_history = pd.DataFrame()

tuning_history = pd.concat(
    [tuning_history, pd.DataFrame([trial_record])], ignore_index=True
)
tuning_history.to_csv(TUNING_HISTORY_PATH, index=False)

# Muc tieu chinh la so pill duoc accepted/identified; it ambiguous/unknown hon la tie-break.
ranking = tuning_history.sort_values(
    by=[
        'identified_count', 'ambiguous_count', 'unknown_count',
        'insufficient_visual_evidence_count', 'mean_top_candidate_score',
        'detected_instances',
    ],
    ascending=[False, True, True, True, False, False],
    na_position='last',
).reset_index(drop=True)

print('Saved tuning history:', TUNING_HISTORY_PATH)
display(ranking)

best_trial = ranking.iloc[0]
print('\n===== BEST SEGMENTATION TRIAL =====')
print('run_name:', best_trial['run_name'])
print(
    'identified:',
    f"{int(best_trial['identified_count'])}/{int(best_trial['detected_instances'])}",
)
print('overrides:', best_trial['overrides_json'])
print('output_dir:', best_trial['output_dir'])


In [ ]:
# DUAL AUTO-TUNE: Tune kep Segmentation (Crop) + OCR (PaddleOCR)
import gc
import itertools
import random
import json
import yaml
import difflib
import pandas as pd
from pathlib import Path

AUTO_TUNING_MAX_TRIALS = 30
AUTO_TUNING_SEED = 42
AUTO_TUNING_ROOT = BASE_OUTPUT_DIR / 'automatic_segmentation_tuning'
AUTO_TUNING_ROOT.mkdir(parents=True, exist_ok=True)

# THAY DOI CAC THAM SO O DAY
AUTO_TUNING_GRID = {
    # SEGMENTATION PARAMS
    'bbox_padding_ratio': [0.06, 0.08, 0.12],
    'color_mask_erosion_ratio': [0.02, 0.04, 0.06],
    
    # OCR PARAMS (co chu ocr_ o dau)
    'ocr_det_db_thresh': [0.1, 0.2, 0.3],
    'ocr_det_db_unclip_ratio': [1.5, 2.0, 2.5],
    'ocr_min_usable_confidence': [0.30, 0.50],
}

# Ground truth map
ground_truth_labels = {
    'pill_003': '54868-5095',
    'pill_004': '54348-857',
    'pill_001': '61919-448',
    'pill_002': '72789-401'
}

json_path = Path('/kaggle/input/datasets/nnphuchcmus/drug-label/drug_appearances.json')
if not json_path.exists():
    json_path = Path('D:/Dowloads/drug_appearances.json')

if not json_path.exists():
    print(f"KHONG TIM THAY JSON TAI: {json_path}")
else:
    with open(json_path, 'r', encoding='utf-8') as f:
        drug_db_dict = {item['product_code']: item for item in json.load(f)}
        
    def _non_null_overrides(overrides):
        return {name: value for name, value in overrides.items() if value is not None}

    def _run_automatic_trial(trial_index, overrides):
        trial_name = f'trial_{trial_index:02d}'
        trial_output = AUTO_TUNING_ROOT / trial_name
        active_overrides = _non_null_overrides(overrides)
        
        # Phan chia param
        seg_overrides = {k: v for k, v in active_overrides.items() if not k.startswith('ocr_')}
        ocr_overrides = {k.replace('ocr_', ''): v for k, v in active_overrides.items() if k.startswith('ocr_')}
        
        print(f'\n===== AUTO TRIAL {trial_index} =====')
        print(f'Seg Params: {seg_overrides}')
        print(f'OCR Params: {ocr_overrides}')

        trial_request = request.model_copy(update={'request_id': f'{REQUEST_ID}_{trial_name}'})

        # MODULE 1
        segmentation_config = SegmentationConfig.from_yaml(REPO_DIR / 'configs' / 'inference' / 'segmentation.yaml').with_weights_path(SEGMENTATION_WEIGHTS).with_output_dir(trial_output)
        segmentation_config = replace(segmentation_config, **seg_overrides)
        segmentation_predictor = SegmentationPredictor(config=segmentation_config)
        segmentation_artifacts = segmentation_predictor.predict_with_artifacts(trial_request)
        segmentation_output = segmentation_artifacts.output

        if not segmentation_output.instances:
            return {'run_name': trial_name, 'status': 'success', 'detected_instances': 0, 'score': 0, 'details': 'No instances', 'overrides_json': json.dumps(active_overrides, sort_keys=True)}

        # MODULE 2
        attribute_config = AttributeInferenceConfig.from_yaml(REPO_DIR / 'configs' / 'inference' / 'attribute.yaml').resolve_paths(REPO_DIR)
        attribute_config = replace(attribute_config, weights_path=ATTRIBUTE_ARTIFACT_DIR / 'best.pt', label_mapping_path=ATTRIBUTE_ARTIFACT_DIR / 'label_mapping.json', color_thresholds_path=ATTRIBUTE_ARTIFACT_DIR / 'optimal_thresholds.json', model_config_path=ATTRIBUTE_ARTIFACT_DIR / 'model_config.yaml', output_dir=trial_output)
        attribute_predictor_trial = AttributePredictor(config=attribute_config)
        attribute_outputs_trial = []
        for instance in segmentation_output.instances:
            attribute_request = AttributeInferenceRequest(
                request_id=segmentation_output.request_id, session_id=segmentation_output.session_id, image_id=segmentation_output.image_id, instance_id=instance.instance_id, instance_token=instance.instance_token, crop_path=instance.color_crop_path, color_crop_path=instance.color_crop_path, shape_crop_path=instance.shape_crop_path, mask_path=instance.mask_path
            )
            attribute_outputs_trial.append(attribute_predictor_trial.predict(attribute_request))

        del segmentation_predictor, attribute_predictor_trial
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        # SETUP OCR CONFIG OVERRIDE
        ocr_output_dir_trial = trial_output / 'predictions' / 'ocr'
        ocr_request_dir_trial = trial_output / 'requests' / 'ocr'
        ocr_output_dir_trial.mkdir(parents=True, exist_ok=True)
        ocr_request_dir_trial.mkdir(parents=True, exist_ok=True)
        
        # Doc config ocr yaml cu va ghi de
        with open(REPO_DIR / 'configs' / 'inference' / 'ocr.yaml', 'r') as f:
            ocr_cfg_dict = yaml.safe_load(f)
            
        if 'det_db_thresh' in ocr_overrides:
            ocr_cfg_dict['model']['det_db_thresh'] = float(ocr_overrides['det_db_thresh'])
        if 'det_db_unclip_ratio' in ocr_overrides:
            ocr_cfg_dict['model']['det_db_unclip_ratio'] = float(ocr_overrides['det_db_unclip_ratio'])
        if 'min_usable_confidence' in ocr_overrides:
            ocr_cfg_dict['pipeline']['min_usable_confidence'] = float(ocr_overrides['min_usable_confidence'])
            
        trial_ocr_config_path = ocr_request_dir_trial / 'ocr_tuning.yaml'
        with open(trial_ocr_config_path, 'w') as f:
            yaml.dump(ocr_cfg_dict, f)

        # MODULE 3 (OCR)
        ocr_results_trial = []
        for instance in segmentation_output.instances:
            instance_id = instance.instance_id
            req_path = ocr_request_dir_trial / f'{instance_id}_req.json'
            payload = {'request_id': segmentation_output.request_id, 'session_id': segmentation_output.session_id, 'image_id': segmentation_output.image_id, 'instance_id': instance_id, 'instance_token': instance.instance_token, 'crop_path': instance.ocr_crop_path, 'mask_path': instance.mask_path}
            req_path.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
            
            completed = subprocess.run([str(OCR_PYTHON), str(OCR_RUNNER), '--request', str(req_path), '--config', str(trial_ocr_config_path), '--output-dir', str(ocr_output_dir_trial)], cwd=str(REPO_DIR), capture_output=True, text=True)
            
            inst_dir = ocr_output_dir_trial / ocr_artifact_directory_name(segmentation_output.request_id) / ocr_artifact_directory_name(segmentation_output.image_id) / ocr_artifact_directory_name(instance_id)
            schema_path = inst_dir / f'{ocr_artifact_directory_name(instance_id)}_ocr_schema.json'
            if schema_path.is_file():
                ocr_results_trial.append(json.loads(schema_path.read_text(encoding='utf-8')))

        # MODULE 4
        cv_pipeline_config_trial = CVPipelineConfig.from_yaml(REPO_DIR / 'configs' / 'inference' / 'cv_pipeline.yaml').with_output_dir(trial_output)
        cv_pipeline_artifacts_trial = CVPipelineAssembler(config=cv_pipeline_config_trial).predict_with_artifacts(
            CVPipelineInput(segmentation_output=segmentation_output, attribute_outputs=attribute_outputs_trial, ocr_outputs=ocr_results_trial)
        )
        cv_output_trial = cv_pipeline_artifacts_trial.output.model_dump(mode='json')

        # EVALUATION JSON
        score = 0
        details = []
        for pill in cv_output_trial.get('pills', []):
            iid = pill['instance_id']
            if iid not in ground_truth_labels: continue
            gt_info = drug_db_dict.get(ground_truth_labels[iid])
            if not gt_info: continue
            
            pred_shape = str(pill.get('shape', {}).get('label', '')).upper()
            pred_color = str(pill.get('color', {}).get('primary', '')).upper()
            pred_imprint = str(pill.get('imprint', {}).get('raw', '')).upper()
            if pred_imprint == 'UNKNOWN': pred_imprint = ''
            
            gt_shape = str(gt_info.get('shape') or '').upper()
            gt_color = str(gt_info.get('primary_color') or '').upper()
            gt_imprint = str(gt_info.get('imprint_normalized') or '').upper()
            
            is_s = 1.0 if pred_shape == gt_shape else 0.0
            is_c = 1.0 if pred_color == gt_color else 0.0
            
            score_i = 0.0
            if gt_imprint == pred_imprint:
                score_i = 1.0
            elif gt_imprint and pred_imprint:
                sm = difflib.SequenceMatcher(None, gt_imprint, pred_imprint)
                match = sm.find_longest_match(0, len(gt_imprint), 0, len(pred_imprint))
                score_i = match.size / len(gt_imprint)
            
            pts = is_s + is_c + score_i
            score += pts
            details.append(f'{iid}: S={is_s}, C={is_c}, I={score_i:.2f}')
            
        print(f'-> Trial Score: {score:.2f}/{len(ground_truth_labels)*3}')
        return {
            'run_name': trial_name,
            'status': 'success',
            'detected_instances': len(segmentation_output.instances),
            'score': score,
            'details': ' | '.join(details),
            'overrides_json': json.dumps(active_overrides, sort_keys=True),
            'output_dir': str(trial_output)
        }

    all_combinations = [dict(zip(AUTO_TUNING_GRID, values)) for values in itertools.product(*AUTO_TUNING_GRID.values())]
    
    baseline_trial = {}
    for name in AUTO_TUNING_GRID:
        baseline_trial[name] = SEGMENTATION_OVERRIDES.get(name, None) if not name.startswith('ocr_') else None
        
    candidate_combinations = [combo for combo in all_combinations if combo != baseline_trial]
    random.Random(AUTO_TUNING_SEED).shuffle(candidate_combinations)
    selected_trials = [baseline_trial] + candidate_combinations[:AUTO_TUNING_MAX_TRIALS - 1]
    
    print(f'Automatic tuning trials: {len(selected_trials)}')
    automatic_records = []
    
    for trial_index, trial_overrides in enumerate(selected_trials, start=1):
        try:
            automatic_records.append(_run_automatic_trial(trial_index, trial_overrides))
        except Exception as error:
            print(f'Trial {trial_index} failed: {error}')
            automatic_records.append({'run_name': f'trial_{trial_index:02d}', 'status': 'failed', 'error': repr(error), 'overrides_json': json.dumps(_non_null_overrides(trial_overrides), sort_keys=True)})

    automatic_history = pd.DataFrame(automatic_records)
    automatic_history_path = BASE_OUTPUT_DIR / 'automatic_segmentation_tuning_history_json.csv'
    automatic_history.to_csv(automatic_history_path, index=False)
    
    successful_trials = automatic_history[automatic_history['status'] == 'success'].copy()
    if successful_trials.empty:
        print('No automatic tuning trial completed successfully.')
    else:
        best_automatic_trial = successful_trials.sort_values(by=['score', 'detected_instances'], ascending=[False, False], na_position='last').iloc[0]
        
        print('\n===== DUAL AUTO TUNING RESULT =====')
        print('best run:', best_automatic_trial['run_name'])
        print('score:', f"{best_automatic_trial['score']:.2f}/{len(ground_truth_labels)*3}")
        print('best overrides:', best_automatic_trial['overrides_json'])
        print('best output:', best_automatic_trial['output_dir'])
        
        display(successful_trials.sort_values(by=['score', 'detected_instances'], ascending=[False, False], na_position='last'))
